## 1. Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Clone Your Repository (or upload files)

In [ ]:
# Option 1: Clone from GitHub
!git clone https://github.com/Raynergy-svg/ml_engine.git
%cd ml_engine

In [ ]:
# Option 2: Upload files manually (uncomment if not using git)
# from google.colab import files
# !mkdir -p ml_engine
# %cd ml_engine
# uploaded = files.upload()  # Upload your files here

## 3. Install Dependencies

In [ ]:
!pip install -q torch numpy pandas scikit-learn pyyaml tqdm rich matplotlib seaborn

## 4. Update Config for GPU Training

In [ ]:
import yaml

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Update for GPU and faster training
config['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
config['batch_size'] = 128  # Larger batch size for GPU
config['epochs'] = 100  # Adjust as needed
config['auto_resume'] = True  # Enable auto-resume
config['mixed_precision'] = True  # Enable mixed precision for faster training
config['early_stopping_patience'] = 20

# Save updated config
with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

print("Config updated:")
print(f"  Device: {config['device']}")
print(f"  Batch size: {config['batch_size']}")
print(f"  Epochs: {config['epochs']}")

## 5. Train the Model

In [ ]:
from ml_engine_enhanced import EnhancedMLEngine
from data_loader import MarketDataLoader
from utils import load_config
import numpy as np

# Load config
config = load_config('config.yaml')

# Load market data
print("Loading market data...")
data_loader = MarketDataLoader(config)
df = data_loader.load_csv()

if df is None or df.empty:
    raise ValueError("No market data found. Upload CSV files to market_data/ folder.")

# Preprocess data
print(f"Loaded {len(df)} data points from {df['symbol'].nunique()} tickers")
X, y = data_loader.preprocess(df)

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")

# Update model config with correct input size
if 'model' not in config:
    config['model'] = {}
config['model']['input_size'] = X.shape[2]

# Split into train/val
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"\nTraining samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

# Create engine and train
print("\nInitializing ML Engine...")
engine = EnhancedMLEngine(config)

print("\nStarting training...")
result = engine.train(
    X_train, y_train,
    X_val, y_val,
    epochs=config['epochs']
)

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Resumed: {result.get('resumed', False)}")
print(f"Total epochs: {result.get('total_epochs', 'N/A')}")
print(f"Best validation loss: {result['best_val_loss']:.6f}")
print(f"Final train loss: {result['train_losses'][-1]:.6f}")
print(f"Final val loss: {result['val_losses'][-1]:.6f}")

## 6. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(result['train_losses'], label='Train Loss')
plt.plot(result['val_losses'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(result['val_losses'], label='Val Loss', color='orange')
plt.axhline(y=result['best_val_loss'], color='r', linestyle='--', label='Best Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 7. Evaluate Model

In [ ]:
# Evaluate on validation set
metrics = engine.evaluate(X_val, y_val)

print("Validation Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.6f}")

## 8. Download Trained Model

In [ ]:
from google.colab import files
import shutil
import os

# Create a zip file with all trained models
shutil.make_archive('trained_models', 'zip', 'trained_data/models')

# Download
files.download('trained_models.zip')

print("\nDownload complete! Extract and place in your local trained_data/models/ folder.")

## 9. Continue Training (Optional)

Run this cell to train for additional epochs (will auto-resume from best checkpoint):

In [ ]:
# Train for 50 more epochs
result2 = engine.train(
    X_train, y_train,
    X_val, y_val,
    epochs=50
)

print(f"\nContinued training:")
print(f"Total epochs now: {result2.get('total_epochs', 'N/A')}")
print(f"Best validation loss: {result2['best_val_loss']:.6f}")

---

## Tips for Free GPU Training

**Google Colab Free Tier:**
- GPU: Tesla T4 (16GB)
- Session timeout: ~12 hours
- Auto-disconnect after ~90 min idle

**To maximize free GPU time:**
1. Keep the tab active (prevents idle disconnect)
2. Use auto-resume (enabled above) so you can restart if disconnected
3. Download checkpoints periodically
4. Consider Colab Pro ($10/mo) for longer sessions and better GPUs

**Alternatives:**
- **Kaggle Notebooks**: 30 hrs/week free GPU
- **Paperspace Gradient**: Free tier with 6hr sessions
- **Lightning AI**: Free tier available